In [ ]:
!pip install transformers datasets sentencepiece evaluate

In [ ]:
import json
import re
from tqdm import tqdm
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving mintaka_dev.json to mintaka_dev.json
Saving mintaka_test.json to mintaka_test.json
Saving mintaka_train.json to mintaka_train.json


In [ ]:
def extract_answer(item):
    ans = item.get("answer", {})

    if isinstance(ans, dict) and "answer" in ans and ans["answer"]:
        obj = ans["answer"][0]

        if isinstance(obj, dict):
            if "label" in obj and isinstance(obj["label"], dict):
                if obj["label"].get("en"):
                    return str(obj["label"]["en"])

            if "name" in obj:
                return str(obj["name"])

        return str(obj)

    if isinstance(ans, dict) and "mention" in ans:
        return str(ans["mention"])

    return ""


def load_mintaka(path):
    with open(path) as f:
        data = json.load(f)

    questions, answers = [], []

    for item in data:
        questions.append(item["question"])
        answers.append(extract_answer(item))

    return questions, answers


train_q, train_a = load_mintaka("mintaka_train.json")
dev_q, dev_a     = load_mintaka("mintaka_dev.json")
test_q, test_a   = load_mintaka("mintaka_test.json")

In [ ]:
def create_tuples_no_ned(questions, answers):
    inputs, labels = [], []

    for q, ans in zip(questions, answers):
        inputs.append(f"question: {q}")
        labels.append(ans)

    return inputs, labels


train_inp, train_lab = create_tuples_no_ned(train_q, train_a)
dev_inp, dev_lab     = create_tuples_no_ned(dev_q, dev_a)
test_inp, test_lab   = create_tuples_no_ned(test_q, test_a)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def make_dataset(inputs, labels):
    ds = Dataset.from_dict({
        "input_text": inputs,
        "target_text": labels
    })

    def tokenize(batch):
        model_inputs = tokenizer(
            batch["input_text"],
            truncation=True,
            padding="max_length",
            max_length=512
        )

        labels_tok = tokenizer(
            batch["target_text"],
            truncation=True,
            padding="max_length",
            max_length=32
        )

        model_inputs["labels"] = labels_tok["input_ids"]
        return model_inputs

    ds = ds.map(tokenize, batched=True)

    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )

    return ds


train_dataset = make_dataset(train_inp, train_lab)
dev_dataset   = make_dataset(dev_inp, dev_lab)
test_dataset  = make_dataset(test_inp, test_lab)

Map:   0%|          | 0/14000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = T5ForConditionalGeneration.from_pretrained("t5-base").to(device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./t5_no_ned",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=3e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.272852,0.246022
2,0.277074,0.245772
3,0.276888,0.245772


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10500, training_loss=0.32588316526867095, metrics={'train_runtime': 3738.166, 'train_samples_per_second': 11.235, 'train_steps_per_second': 2.809, 'total_flos': 2.557623140352e+16, 'train_loss': 0.32588316526867095, 'epoch': 3.0})

In [ ]:
def predict_answer(question):
    inp = f"question: {question}"

    ids = tokenizer(inp, return_tensors="pt").input_ids.to(model.device)
    out = model.generate(ids, max_length=32)

    return tokenizer.decode(out[0], skip_special_tokens=True)


preds = []
for q in tqdm(test_q):
    preds.append(predict_answer(q))

100%|██████████| 4000/4000 [06:33<00:00, 10.17it/s]


In [ ]:
import re

def normalize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return " ".join(text.split())

In [ ]:
def f1_score(pred, gold):
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()

    common = set(pred_tokens) & set(gold_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
hit1 = 0
f1_total = 0

for p, g in zip(preds, test_a):
    if normalize(p) == normalize(g):
        hit1 += 1

    f1_total += f1_score(p, g)

hit1 = hit1 / len(test_a)
f1 = f1_total / len(test_a)

# For single prediction:
hit5 = hit1
mrr = hit1
accuracy = hit1

print("Hit@1:", hit1)
print("Hit@5:", hit5)
print("MRR:", mrr)
print("F1:", f1)
print("Accuracy:", accuracy)

Hit@1: 0.18075
Hit@5: 0.18075
MRR: 0.18075
F1: 0.24377894672865225
Accuracy: 0.18075


In [ ]:
print(train_inp[0])

question: What is the seventh tallest mountain in North America?
